# Check `backend/RAG.py` Output

This notebook runs one end-to-end RAG query against the existing Chroma collection and prints the answer, document sources, graph sources, token usage, and optional graph error.

Before running it, make sure `OPENAI_API_KEY` is available in `/workspace/.env` or in your shell environment. If Neo4j is not running, `RAG.py` will still answer from document retrieval and place the graph issue in `graph_error`.

In [1]:
from pathlib import Path
import json
import os
import sys

from dotenv import load_dotenv


In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=False)

if not (os.getenv("OPENAI_API_KEY") or "").strip():
    raise ValueError("OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.")

print(f"Project root: {PROJECT_ROOT}")


Project root: /workspace


In [ ]:
from backend.RAG import rag

query = "In the sale deed, who sold the property at Kerkstraat 12 and what was the sale price?"

result = rag(
    query=query,
    top_k=3,
)


In [ ]:
print("QUESTION")
print(query)

print("\nANSWER")
print(result["answer"])

print("\nDOCUMENT SOURCES")
print(json.dumps(result["sources"], indent=2, ensure_ascii=False))

print("\nGRAPH SOURCES")
print(json.dumps(result["graph_sources"], indent=2, ensure_ascii=False))

if result.get("graph_error"):
    print("\nGRAPH ERROR")
    print(result["graph_error"])

print("\nRUN INFO")
print(json.dumps({
    "model_used": result["model_used"],
    "embedding_model": result["embedding_model"],
    "reranker_model": result["reranker_model"],
    "collection": result["collection"],
    "top_k": result["top_k"],
    "response_time_seconds": result["response_time_seconds"],
    "prompt_tokens": result["prompt_tokens"],
    "completion_tokens": result["completion_tokens"],
    "total_tokens": result["total_tokens"],
}, indent=2, ensure_ascii=False))


To inspect the exact prompt sent to the chat model, run the optional cell below.

In [ ]:
print(result["prompt"])
